# Gaussian Splatting — Getting Started Notebook

This notebook runs inside the **base** container (via `docker compose up jupyter`).
It walks through:
1. GPU sanity check
2. Loading a trained `.ply` checkpoint
3. Visualizing Gaussian properties (position, scale, opacity)
4. Computing metrics on a rendered output folder

In [ ]:
import torch
import subprocess

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('\nnvidia-smi:', result.stdout.strip())

## Load a Trained Gaussian Cloud (.ply)

In [ ]:
from plyfile import PlyData
import numpy as np
import matplotlib.pyplot as plt

PLY_PATH = '/outputs/my_scene/point_cloud/iteration_30000/point_cloud.ply'

plydata = PlyData.read(PLY_PATH)
v = plydata['vertex']

print(f'Number of Gaussians: {len(v):,}')
print(f'Properties: {[p.name for p in v.properties]}')

In [ ]:
xyz    = np.stack([v['x'], v['y'], v['z']], axis=-1)
opacity = 1 / (1 + np.exp(-v['opacity']))  # sigmoid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(opacity, bins=100, color='steelblue')
axes[0].set_title('Opacity Distribution')
axes[0].set_xlabel('Opacity')

axes[1].scatter(xyz[:, 0], xyz[:, 1], s=0.1, alpha=0.3, c=opacity, cmap='plasma')
axes[1].set_title('XY projection (colored by opacity)')
axes[1].set_aspect('equal')

axes[2].scatter(xyz[:, 0], xyz[:, 2], s=0.1, alpha=0.3, c=opacity, cmap='plasma')
axes[2].set_title('XZ projection (colored by opacity)')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'XYZ range: x=[{xyz[:,0].min():.2f}, {xyz[:,0].max():.2f}]',
      f'y=[{xyz[:,1].min():.2f}, {xyz[:,1].max():.2f}]',
      f'z=[{xyz[:,2].min():.2f}, {xyz[:,2].max():.2f}]')

## Compute PSNR / SSIM / LPIPS on Rendered Images

In [ ]:
import os
from pathlib import Path
from PIL import Image
import torch
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips

RENDER_DIR  = Path('/outputs/my_scene/train/ours_30000/renders')
GT_DIR      = Path('/outputs/my_scene/train/ours_30000/gt')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
psnr_fn  = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_fn  = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips_fn = lpips.LPIPS(net='vgg').to(device)

psnrs, ssims, lp = [], [], []

renders = sorted(RENDER_DIR.glob('*.png'))
for r in renders:
    gt_path = GT_DIR / r.name
    if not gt_path.exists():
        continue
    img_r = torch.from_numpy(np.array(Image.open(r)) / 255.).permute(2,0,1).float().unsqueeze(0).to(device)
    img_g = torch.from_numpy(np.array(Image.open(gt_path)) / 255.).permute(2,0,1).float().unsqueeze(0).to(device)
    psnrs.append(psnr_fn(img_r, img_g).item())
    ssims.append(ssim_fn(img_r, img_g).item())
    lp.append(lpips_fn(img_r * 2 - 1, img_g * 2 - 1).item())

print(f'Images evaluated: {len(psnrs)}')
print(f'PSNR:  {np.mean(psnrs):.2f} dB')
print(f'SSIM:  {np.mean(ssims):.4f}')
print(f'LPIPS: {np.mean(lp):.4f}')

## Prune Low-Opacity Gaussians and Save

In [ ]:
OPACITY_THRESHOLD = 0.05

mask = opacity > OPACITY_THRESHOLD
print(f'Before pruning: {len(v):,} Gaussians')
print(f'After pruning (opacity > {OPACITY_THRESHOLD}): {mask.sum():,} Gaussians')
print(f'Removed: {(~mask).sum():,} ({100*(~mask).mean():.1f}%)')